# Regional Weather Preparation

This notebook:

1. loads census state centroids and population weights;
2. selects states for an EIA storage region;
3. downloads daily mean temperatures from Open-Meteo;
4. calculates HDD/CDD at state level;
5. aggregates to a population-weighted regional weather index;
6. exports processed datasets for modeling.

In [1]:
from pathlib import Path

import pandas as pd

from gas_forecast.data.export import save_versioned_parquet
from gas_forecast.data.regions import region_slug, region_states
from gas_forecast.data.weather import (
    aggregate_population_weighted_weather,
    calculate_hdd_cdd,
    fetch_all_state_temperatures,
    load_census_state_locations,
    migrate_weather_chunk_cache,
    prepare_weather_model_data,
    select_weather_locations,
    validate_state_daily_weather,
)

In [2]:
REGION = "R48"  # R48, R01, R02, R03, R04, R05

CACHE_DIR = Path("../data/cache")
PROCESSED_DIR = Path("../data/processed")
WEATHER_CACHE_DIR = CACHE_DIR / "weather"
LEGACY_WEATHER_CACHE_DIR = Path("../data/raw/weather_cache")

REGION_SLUG = region_slug(REGION)
STORAGE_PATH = PROCESSED_DIR / f"{REGION_SLUG}_weekly_storage_latest.parquet"

# One-time migration from legacy hash-keyed chunk cache (safe to re-run).
migrate_weather_chunk_cache(LEGACY_WEATHER_CACHE_DIR, WEATHER_CACHE_DIR)

In [3]:
storage = pd.read_parquet(STORAGE_PATH)

START_DATE = storage["date"].min().strftime("%Y-%m-%d")
END_DATE = storage["date"].max().strftime("%Y-%m-%d")

In [4]:
locations = load_census_state_locations()
locations = select_weather_locations(locations, REGION)

locations.head()

,STATEFP,STNAME,POPULATION,LATITUDE,LONGITUDE,WEIGHT
0,01,Alabama,5024279,33.016191,-86.753353,0.015259
1,04,Arizona,7151502,33.371388,-111.882468,0.021720
2,05,Arkansas,3011524,35.199251,-92.713212,0.009146
3,06,California,39538223,35.491035,-119.347852,0.120082
4,08,Colorado,5773714,39.534747,-105.185361,0.017535


In [5]:
state_daily_weather = fetch_all_state_temperatures(
    locations=locations,
    start_date=START_DATE,
    end_date=END_DATE,
    cache_dir=WEATHER_CACHE_DIR,
    incremental=True,
    pause_seconds=3.0,
)

[1/130] 2014-07-11 to 2014-12-31 | Alabama, Arizona, Arkansas, California, Colorado
[2/130] 2014-07-11 to 2014-12-31 | Connecticut, Delaware, District of Columbia, Florida, Georgia
[3/130] 2014-07-11 to 2014-12-31 | Idaho, Illinois, Indiana, Iowa, Kansas
[4/130] 2014-07-11 to 2014-12-31 | Kentucky, Louisiana, Maine, Maryland, Massachusetts
[5/130] 2014-07-11 to 2014-12-31 | Michigan, Minnesota, Mississippi, Missouri, Montana
[6/130] 2014-07-11 to 2014-12-31 | Nebraska, Nevada, New Hampshire, New Jersey, New Mexico
[7/130] 2014-07-11 to 2014-12-31 | New York, North Carolina, North Dakota, Ohio, Oklahoma
[8/130] 2014-07-11 to 2014-12-31 | Oregon, Pennsylvania, Rhode Island, South Carolina, South Dakota
[9/130] 2014-07-11 to 2014-12-31 | Tennessee, Texas, Utah, Vermont, Virginia


KeyboardInterrupt: 

In [ ]:
validate_state_daily_weather(
    state_daily_weather,
    expected_states=region_states(REGION),
)

state_daily_weather.head()

In [ ]:
save_versioned_parquet(
    state_daily_weather,
    PROCESSED_DIR,
    f"{REGION_SLUG}_state_daily_weather",
    save_latest=True,
)

In [ ]:
state_degrees = calculate_hdd_cdd(state_daily_weather)
regional_weather = aggregate_population_weighted_weather(state_degrees)

regional_weather.head()

In [ ]:
regional_weather_model = prepare_weather_model_data(regional_weather, REGION)

save_versioned_parquet(
    regional_weather_model,
    PROCESSED_DIR,
    f"{REGION_SLUG}_daily_weather",
    save_latest=True,
)